# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suyashverma0/flyrank-ml-workspace2/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
import pandas as pd
from pathlib import Path
import subprocess

repo_path = Path("/content/flyrank-ml-workspace2")

if not repo_path.exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/suyashverma0/flyrank-ml-workspace2.git",
        str(repo_path)
    ], check=True)

matches = list(repo_path.rglob("content_refresh_anonymized.csv"))

if not matches:
    raise FileNotFoundError("Dataset CSV nahi mili.")

data_path = matches[0]
df = pd.read_csv(data_path)

print("Dataset loaded:", df.shape)
df.head()

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1: Verify available data

required_columns = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "avg_position"
]

print("Column availability:")
for col in required_columns:
    print(col, ":", col in df.columns)

print("\nSearch volume vs impressions correlation:")

correlation = df[
    ["search_volume", "impressions_90d"]
].corr().iloc[0, 1]

print("Correlation:", round(correlation, 4))

print("\nRows available for position analysis:")

valid_rows = df[
    ["clicks_90d", "impressions_90d", "avg_position"]
].notna().all(axis=1).sum()

print("Valid rows:", valid_rows)

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.base import clone

# Function to evaluate one split
def evaluate_model_split(train_idx, test_idx, split_name):

    X_tr = X.iloc[train_idx].copy()
    X_te = X.iloc[test_idx].copy()

    y_tr = y.iloc[train_idx].copy()
    y_te = y.iloc[test_idx].copy()

    current_model = clone(model)
    current_model.fit(X_tr, y_tr)

    probabilities = current_model.predict_proba(X_te)[:, 1]

    ranking = pd.Series(
        probabilities,
        index=X_te.index
    ).sort_values(ascending=False).index

    return {
        "Split": split_name,
        "Train Rows": len(train_idx),
        "Test Rows": len(test_idx),
        "Precision@10": precision_at_k(ranking, y_te, 10),
        "Precision@50": precision_at_k(ranking, y_te, 50),
        "Precision@100": precision_at_k(ranking, y_te, 100),
        "Test Base Rate": y_te.mean()
    }


# BEFORE: Random row split
indices = np.arange(len(X))

random_train_idx, random_test_idx = train_test_split(
    indices,
    test_size=0.25,
    random_state=42,
    stratify=y
)

random_result = evaluate_model_split(
    random_train_idx,
    random_test_idx,
    "Random Row Split"
)


# AFTER: Client-grouped split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

group_train_idx, group_test_idx = next(
    splitter.split(X, y, groups=groups)
)

group_result = evaluate_model_split(
    group_train_idx,
    group_test_idx,
    "Client Grouped Split"
)


# Check client overlap
random_overlap = len(
    set(groups.iloc[random_train_idx])
    & set(groups.iloc[random_test_idx])
)

group_overlap = len(
    set(groups.iloc[group_train_idx])
    & set(groups.iloc[group_test_idx])
)

print("Random split client overlap:", random_overlap)
print("Grouped split client overlap:", group_overlap)

assert group_overlap == 0

# Comparison table
validation_results = pd.DataFrame([
    random_result,
    group_result
])

display(validation_results.round(4))

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Final feature list from Week-5
final_features = numeric_features + categorical_features

# Forbidden columns
forbidden_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_proxy",
    "is_declining_label"
]

# Performance fields excluded from metadata-only model
performance_columns = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

# Leakage report
leakage_report = pd.DataFrame({
    "Feature": final_features,
    "Forbidden": [
        col in forbidden_columns
        for col in final_features
    ],
    "Performance Field": [
        col in performance_columns
        for col in final_features
    ]
})

display(leakage_report)

# Assertions
assert not set(final_features).intersection(forbidden_columns)
assert not set(final_features).intersection(performance_columns)

assert "trend_direction" not in X.columns
assert "trend_pct" not in X.columns
assert "content_id" not in X.columns
assert "client_id" not in X.columns

print("Total features:", len(final_features))
print("Forbidden feature overlap:",
      len(set(final_features).intersection(forbidden_columns)))

print("Performance feature overlap:",
      len(set(final_features).intersection(performance_columns)))

print("Leakage audit: PASSED")

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
claim_audit = pd.DataFrame([
    {
        "Type": "Original Claim",
        "Statement": (
            "The Logistic Regression model can identify declining "
            "content and predict which pages need refreshing."
        )
    },
    {
        "Type": "Rewritten Claim",
        "Statement": (
            "The model ranks content items by their estimated likelihood "
            "of matching a historical declining-content proxy. "
            "Its client-grouped Precision@K supports decision-making "
            "for further content review, not proof of refresh success."
        )
    }
])

display(claim_audit)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.